# Exploratory Data Analysis
Market Signal Forecaster — AAPL, MSFT, SPY, QQQ (2018–2024)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings("ignore")

tickers = ["AAPL", "MSFT", "SPY", "QQQ"]
data = {t: pd.read_parquet(f"data/raw/{t}.parquet") for t in tickers}
print({t: df.shape for t, df in data.items()})

## Price History

In [ ]:
fig = go.Figure()
for ticker, df in data.items():
    fig.add_trace(go.Scatter(x=df.index, y=df["close"] / df["close"].iloc[0],
                             name=ticker, mode="lines"))
fig.update_layout(title="Normalized Price (base=1.0)", template="plotly_dark",
                  xaxis_title="Date", yaxis_title="Cumulative Return (×)",
                  height=400)
fig.show()

## Return Distributions

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=tickers)
for i, (ticker, df) in enumerate(data.items()):
    r, c = divmod(i, 2)
    log_ret = np.log(df["close"] / df["close"].shift(1)).dropna() * 100
    fig.add_trace(go.Histogram(x=log_ret, name=ticker, nbinsx=80,
                               marker_color=["#89b4fa","#a6e3a1","#f38ba8","#fab387"][i]),
                 row=r+1, col=c+1)
    print(f"{ticker}: mean={log_ret.mean():.4f}% | std={log_ret.std():.4f}% | skew={log_ret.skew():.3f} | kurt={log_ret.kurt():.3f}")
fig.update_layout(title="Daily Log Return Distributions (%)", template="plotly_dark", height=500)
fig.show()

## Volatility Regimes (Rolling 30-day)

In [ ]:
fig = go.Figure()
for ticker, df in data.items():
    log_ret = np.log(df["close"] / df["close"].shift(1))
    vol = log_ret.rolling(30).std() * np.sqrt(252) * 100
    fig.add_trace(go.Scatter(x=vol.index, y=vol, name=ticker, mode="lines"))
fig.add_hline(y=20, line_dash="dot", line_color="gray", annotation_text="20% threshold")
fig.update_layout(title="Annualized 30-Day Realized Volatility (%)", template="plotly_dark",
                  xaxis_title="Date", yaxis_title="Volatility (%)", height=400)
fig.show()

## Correlation Matrix

In [ ]:
returns = pd.DataFrame({t: np.log(df["close"]/df["close"].shift(1))
                        for t, df in data.items()}).dropna()
corr = returns.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                title="Return Correlation Matrix", template="plotly_dark")
fig.show()
print(corr)